In [55]:
import xarray as xr
import pandas as pd
import glob
import os
import math

import numpy as np
import re

In [56]:
ppes = xr.open_dataset("/glade/work/qingyuany/repo_data/spatialtuning/simv4_iteration1.nc")
obs = xr.open_dataset("/glade/work/qingyuany/repo_data/spatialtuning/obs.nc")

In [57]:

obs_dict = {"SWCF": "swcrf_toa", "LWCF": "lwcrf_toa", "TGCLDLWP": "clwp", "TMQ": "pwv",
          "CLDTOT_ISCCP": "clt_isccp", "FLUT": "olr", "PRECT": "pr","FSNTOA": "swabs_toa"}


In [58]:

lat_bins = np.arange(-85, 86, 5)  # -90 to 90 every 10 degrees
lat_labels = ((lat_bins[:-1] + lat_bins[1:]) / 2).astype(int).astype(str)  # midpoint labels
lat_labels = np.char.add("zonal_lat_",lat_labels)


In [59]:
len(lat_labels)

34

In [60]:
ppe_zonal_list = []
obs_zonal_list = []

for cam_name, obs_name in obs_dict.items():

    ppe_da = ppes[cam_name]
    filter_tf = obs_ds[obs_name].notnull()
    
    ppe_da_filtered = ppe_da.where(filter_tf)
    
    zonal_ppe_temp = ppe_da_filtered.mean(dim  = "lon", skipna = True).groupby_bins("lat",lat_bins, labels = lat_labels).mean(dim = "lat", skipna = True).to_dataframe().unstack(level = 1)
    zonal_ppe_temp.columns.name = None
    zonal_ppe_temp.columns = ["_".join(col) for col in list(zonal_ppe_temp.columns)]

    zonal_obs_temp = obs_ds[obs_name].mean(dim = "lon", skipna = True).groupby_bins("lat",lat_bins, labels = lat_labels).mean(dim = "lat", skipna = True).to_series()
    zonal_obs_temp.index = zonal_ppe_temp.columns

    
    ppe_zonal_list.append(zonal_ppe_temp)
    obs_zonal_list.append(zonal_obs_temp)

ppe_zonal_pd = pd.concat(ppe_zonal_list, axis = 1)
obs_zonal_pd = pd.concat(obs_zonal_list)


In [61]:
#ppe_zonal_pd.to_csv("/glade/work/qingyuany/repo_data/spatialtuning/ppe_zonal_v4_iteration1.csv", index = True)
#obs_zonal_pd.to_csv("/glade/work/qingyuany/repo_data/spatialtuning/obs_zonal_v4_iteration1.csv", index = True)


In [62]:
ppe_zonal_pd.head()

,SWCF_zonal_lat_-82,SWCF_zonal_lat_-77,SWCF_zonal_lat_-72,SWCF_zonal_lat_-67,SWCF_zonal_lat_-62,SWCF_zonal_lat_-57,SWCF_zonal_lat_-52,SWCF_zonal_lat_-47,SWCF_zonal_lat_-42,SWCF_zonal_lat_-37,...,FSNTOA_zonal_lat_37,FSNTOA_zonal_lat_42,FSNTOA_zonal_lat_47,FSNTOA_zonal_lat_52,FSNTOA_zonal_lat_57,FSNTOA_zonal_lat_62,FSNTOA_zonal_lat_67,FSNTOA_zonal_lat_72,FSNTOA_zonal_lat_77,FSNTOA_zonal_lat_82
ppe_ind,,,,,,,,,,,,,,,,,,,,,
1,-1.775323,-3.677040,-8.188253,-26.007094,-55.478150,-73.107625,-77.024634,-73.212884,-64.607686,-54.185431,...,243.736041,217.844014,191.494925,167.835081,146.375524,122.742515,102.767963,87.831024,76.362725,71.129395
3,-1.618221,-3.374800,-8.302124,-26.239062,-55.215606,-72.102811,-75.852205,-72.355755,-63.536108,-53.471915,...,244.924590,219.377728,194.268145,172.547603,151.700622,128.713602,106.657581,88.535728,76.195900,70.599075
4,-1.765804,-3.844527,-8.594140,-27.004102,-56.849576,-74.685015,-78.965654,-77.357821,-74.099307,-67.848767,...,240.021939,215.960871,190.954915,167.326394,146.043029,123.389661,103.185150,86.342926,74.753110,69.292476
5,-2.353393,-4.225666,-8.897521,-27.351285,-56.519404,-73.319435,-76.810730,-74.351819,-69.418165,-63.010395,...,238.841078,213.838495,189.980311,167.055944,144.121663,121.062649,100.773901,85.780473,74.181953,68.465038
6,-1.377978,-3.060643,-7.356543,-24.552219,-52.189145,-68.319704,-71.086715,-68.272482,-62.101780,-52.859389,...,242.563883,217.255606,192.232098,169.226385,148.078477,125.528020,104.687790,88.765484,76.183082,70.918957
